# Covertype + TabFM + XAI (controlled LIME benchmark)

This notebook runs TabFM through the **same controlled LIME protocol** as the Logistic Regression, Random Forest and XGBoost notebooks (`controlled_lime_protocol.py`). It then draws the runtime, fidelity and stability graphs with the shared `plot_controlled_results.py`, so all four models are plotted the same way.

**Running locally on a GPU laptop (not Colab):**
1. Keep this notebook next to `controlled_lime_protocol.py` and `plot_controlled_results.py` (the `experiments/` folder).
2. You need a **CUDA build of PyTorch**. On Windows, a plain `pip install torch` gives a CPU-only build. If the GPU check below fails, install the CUDA wheel from https://pytorch.org/get-started/locally/ first, e.g. `pip install torch --index-url https://download.pytorch.org/whl/cu128`, then restart the kernel.
3. The full benchmark makes about 6 million TabFM predictions, so it takes **hours**. Plug the laptop in and turn off sleep. Progress is checkpointed after every unit: if the run stops (crash, sleep, closed lid), run the benchmark cell again and it picks up where it left off, with identical results.
4. Order: run everything down to the ETA cell. Optionally run the smoke test, then the full benchmark and the graph cells.

**If the GPU check fails with `OSError: [WinError 1455] The paging file is too small`:** Windows ran out of virtual memory while loading the CUDA libraries. This is a Windows setting, not a code bug. Close other programs and restart the kernel. If it keeps happening, raise the page file: *Settings → System → About → Advanced system settings → Performance Settings → Advanced → Virtual memory → Change*. Untick "Automatically manage", choose *System managed size* (or a custom size of at least 16 GB) and restart the laptop.

## 1. Install

In [ ]:
from pathlib import Path

if not Path("tabfm").exists():
    !git clone -q --depth 1 https://github.com/google-research/tabfm.git
%pip install -q "./tabfm[pytorch]" lime scikit-learn scipy matplotlib pandas

In [ ]:
import torch

ALLOW_CPU = False   # TabFM on CPU would take days for the full benchmark

print("PyTorch", torch.__version__, "| CUDA build:", torch.version.cuda)
if torch.cuda.is_available():
    DEVICE = "cuda"
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} ({props.total_memory / 1024**3:.1f} GB)")
elif ALLOW_CPU:
    DEVICE = "cpu"
    print("WARNING: running TabFM on CPU.")
else:
    raise RuntimeError(
        "PyTorch cannot see a CUDA GPU. Most likely the CPU-only torch wheel is installed. "
        "Install the CUDA build from https://pytorch.org/get-started/locally/ "
        "(e.g. pip install torch --index-url https://download.pytorch.org/whl/cu128), "
        "restart the kernel and run again."
    )

## 2. The dataset
Each row is a 30m × 30m patch of forest in Colorado.
**Objective:** predict which of 7 tree types grows there (classification), using map data such as elevation, slope and distance to water.

All model notebooks use the same seeded, stratified 10,000-row sample and 80/20 split from `controlled_lime_protocol.py`.

In [ ]:
import numpy as np
import pandas as pd
import sys

for _directory in (Path.cwd(), Path.cwd() / "XAI" / "experiments", Path.cwd() / "experiments"):
    if (_directory / "controlled_lime_protocol.py").is_file():
        sys.path.insert(0, str(_directory))
        break
else:
    raise FileNotFoundError("Keep controlled_lime_protocol.py beside the experiment notebooks.")

from controlled_lime_protocol import CLASS_NAMES, load_controlled_covtype

X, y, X_train, X_test, y_train, y_test = load_controlled_covtype()
class_names = list(CLASS_NAMES)
df = X.assign(Cover_Type=y)

print("Shared sample shape:", df.shape)
print("Train/test:", X_train.shape, X_test.shape)
print("Class counts:\n", y.value_counts().sort_index())
df.head()

## 3. Load TabFM and check performance

**Context rows.** TabFM is not trained. It reads labelled rows as context every time it predicts. To keep the comparison fair, it gets **the same 8,000 training rows as the other three models**. If the GPU can't hold them, the cell automatically falls back to a smaller stratified sample of those rows and prints the size used. Record that size in the write-up.

**Memory settings (8 GB laptop GPU).**
- `bfloat16` is TabFM's own default precision. It halves the weights from about 6.5 GB to about 3.3 GB.
- `keep_cache_on_device=False` parks the cached context in normal RAM between calls.

Every setting used is saved to `tabfm_run_settings.json` for the methods section.

In [ ]:
import gc
import json

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split
from huggingface_hub import snapshot_download
from safetensors import safe_open
from tabfm import TabFMClassifier
from tabfm import tabfm_v1_0_0_pytorch as tabfm_v1

# bfloat16 is TabFM's default and halves GPU memory (float32 weights alone fill most of an 8 GB card).
TABFM_DTYPE = torch.bfloat16


def load_tabfm_lean(device, dtype):
    """Same weights as tabfm_v1.load(device=..., dtype=...), without the RAM spike.

    tabfm_v1.load builds the whole float32 model in CPU RAM (~6.5 GB), then a bfloat16
    copy, before moving it to the GPU. On Windows that can exhaust the page file
    (OSError 1455). Here the model is built with no memory ("meta" device) and each
    weight is read straight onto the GPU and cast to bfloat16 one tensor at a time.
    """
    folder = Path(snapshot_download(tabfm_v1.HF_REPO_ID, allow_patterns=["classification/*"])) / "classification"
    config = json.loads((folder / "config.json").read_text())
    for key in ("task", "model_type", "version", "framework"):
        config.pop(key, None)
    with torch.device("meta"):
        network = tabfm_v1.TabFM_HF(**config)
    state = {}
    with safe_open(str(folder / "model.safetensors"), framework="pt", device=device) as weights:
        for name in weights.keys():
            tensor = weights.get_tensor(name)
            state[name] = tensor.to(dtype) if tensor.is_floating_point() else tensor
            del tensor
    network.load_state_dict(state, strict=True, assign=True)
    del state
    gc.collect()
    return network.eval()


try:
    tabfm_model = load_tabfm_lean(DEVICE, TABFM_DTYPE)   # downloads ~6.5 GB the first time
except OSError as error:
    if getattr(error, "winerror", None) == 1455 or "1455" in str(error):
        raise OSError(
            "Windows ran out of virtual memory (page file) loading TabFM. Shut down every other "
            "Jupyter kernel (VS Code: 'Jupyter: Shut Down All Kernels' or close the other notebooks), "
            "restart this kernel and run again. See the note at the top of the notebook."
        ) from error
    raise
print(f"TabFM loaded on {DEVICE} in {TABFM_DTYPE}. GPU memory in use: "
      f"{torch.cuda.memory_allocated() / 1024**3:.1f} GB" if DEVICE == "cuda" else "TabFM loaded on CPU.")

N_ESTIMATORS = 8
CONTEXT_SIZES = [len(X_train), 4_000, 2_000]   # try all 8,000 training rows first, like the other models


def fit_tabfm(n_rows):
    """Fit TabFM on n_rows of X_train (all of them, or a stratified sample)."""
    if n_rows >= len(X_train):
        context_X, context_y = X_train, y_train
    else:
        context_X, _, context_y, _ = train_test_split(
            X_train, y_train, train_size=n_rows, stratify=y_train, random_state=42)
    classifier = TabFMClassifier(model=tabfm_model, n_estimators=N_ESTIMATORS, cache_context=True,
                                 keep_cache_on_device=False, random_state=42)
    classifier.fit(context_X, context_y)
    return classifier


model = None
for CONTEXT_ROWS in CONTEXT_SIZES:
    try:
        model = fit_tabfm(CONTEXT_ROWS)
        break
    except torch.cuda.OutOfMemoryError:
        model = None
        gc.collect()
        torch.cuda.empty_cache()
        print(f"GPU out of memory with {CONTEXT_ROWS:,} context rows - trying fewer.")
if model is None:
    raise RuntimeError("TabFM did not fit even with 2,000 context rows. Restart the kernel, close other GPU apps and try again.")

if CONTEXT_ROWS == len(X_train):
    print(f"TabFM context: all {CONTEXT_ROWS:,} training rows (same as the other models).")
else:
    print(f"TabFM context: {CONTEXT_ROWS:,} of {len(X_train):,} training rows (stratified sample).\n"
          "Put this in the write-up: TabFM saw fewer training rows than the other models because of GPU memory.")

FEATURE_NAMES = list(X.columns)
PREDICT_BATCH_SIZE = 256   # rows per TabFM call (a multiple of 128); halves automatically if the GPU runs out of memory


def predict_proba_batched(rows, chunk_rows=None):
    """predict_proba in GPU-sized chunks, every call padded to the same shape.

    TabFM's predict_proba sends every row through in one go, and LIME asks for up to
    10,000 rows at once, which runs out of GPU memory. Rows are predicted independently
    against the cached training context, but in bfloat16 the GPU's rounding depends on
    the tensor shape. Padding every chunk to exactly PREDICT_BATCH_SIZE rows (by repeating
    its last row) keeps the shape fixed, so a row's probabilities do not depend on how
    the rows were chunked or what else was in the batch (checked in the next cell).

    chunk_rows: real rows per call (default PREDICT_BATCH_SIZE); only the sanity check sets it.
    """
    global PREDICT_BATCH_SIZE
    rows = np.asarray(rows)
    outputs = []
    start = 0
    while start < len(rows):
        size = min(chunk_rows or PREDICT_BATCH_SIZE, PREDICT_BATCH_SIZE)
        real = rows[start:start + size]
        padded = np.concatenate([real, np.repeat(real[-1:], PREDICT_BATCH_SIZE - len(real), axis=0)])
        try:
            probabilities = model.predict_proba(pd.DataFrame(padded, columns=FEATURE_NAMES))
            outputs.append(np.asarray(probabilities, dtype=float)[:len(real)])
            start += len(real)
        except torch.cuda.OutOfMemoryError:
            gc.collect()
            torch.cuda.empty_cache()
            if PREDICT_BATCH_SIZE == 1:
                raise
            PREDICT_BATCH_SIZE = max(1, PREDICT_BATCH_SIZE // 2)
            print(f"GPU out of memory - batch size reduced to {PREDICT_BATCH_SIZE}")
    return np.vstack(outputs)


class BatchedTabFM(ClassifierMixin, BaseEstimator):
    """The fitted TabFM model, but predicting in GPU-sized chunks.

    Used by permutation importance and the held-out evaluation, which would otherwise
    send hundreds of rows to the GPU in one call.
    """

    def fit(self, X=None, y=None):   # already fitted above; present so scikit-learn accepts it
        return self

    @property
    def classes_(self):
        return model.classes_

    def predict_proba(self, rows):
        return predict_proba_batched(np.asarray(rows))

    def predict(self, rows):
        return model.classes_[self.predict_proba(rows).argmax(axis=1)]


batched_model = BatchedTabFM()

y_pred = batched_model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, labels=list(range(7)), target_names=class_names, zero_division=0))

In [ ]:
# Sanity check: a row's probabilities must not depend on how the rows are chunked or
# what else is in the batch (LIME sends thousands of perturbed rows per explanation).
check_rows = X_test.iloc[:200].to_numpy()
together = predict_proba_batched(check_rows)                    # 200 rows in one call
chunked = predict_proba_batched(check_rows, chunk_rows=37)      # deliberately awkward chunks
order = np.random.default_rng(0).permutation(len(check_rows))
shuffled = np.empty_like(together)
shuffled[order] = predict_proba_batched(check_rows[order])     # same rows, different batch-mates

difference = max(np.abs(together - chunked).max(), np.abs(together - shuffled).max())
print(f"Max probability difference across chunkings/orderings: {difference:.2e}")
assert difference < 1e-3, "Chunking changes TabFM's output - tell the group before using these results."
if difference > 0:
    print("Small non-zero differences are bfloat16 GPU rounding; mention this in the write-up.")


## 4. XAI 1: Permutation feature importance (global)
Shuffle one feature at a time and see how much accuracy drops. A bigger drop means the model relies on that feature more.

In [ ]:
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt

# Uses the batched wrapper: 54 features x 3 repeats = 162 predictions of 500 rows each.
result = permutation_importance(batched_model, X_test[:500], y_test[:500], n_repeats=3,
                                scoring="accuracy", random_state=42)
importance = pd.Series(result.importances_mean, index=X.columns).sort_values()

importance.tail(10).plot.barh(title="Top 10 features (permutation importance)")
plt.xlabel("Drop in accuracy")
plt.tight_layout()
plt.show()

## 5. XAI 2: LIME (local)
Explain why the model made its prediction for one specific patch of forest (illustration only; the research numbers come from the controlled benchmark below).

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

lime_background = X_train.sample(n=min(2000, len(X_train)), random_state=42)
categorical_indices = list(range(10, X.shape[1]))
explainer = LimeTabularExplainer(
    lime_background.values,
    feature_names=FEATURE_NAMES,
    class_names=class_names,
    categorical_features=categorical_indices,
    categorical_names={i: ["0", "1"] for i in categorical_indices},
    mode="classification",
    random_state=42,
)

row = X_test.iloc[0]
print("Actual:", class_names[y_test.iloc[0]], "| Predicted:", class_names[int(model.predict(X_test.iloc[[0]])[0])])

explanation = explainer.explain_instance(row.values, predict_proba_batched, num_features=10,
                                         num_samples=1000, top_labels=1)
explanation.as_pyplot_figure(label=explanation.top_labels[0])
plt.tight_layout()
plt.show()

## Controlled LIME cost/quality comparison

The earlier single-row plot is illustrative only. The research comparison uses the same test cases, LIME background, perturbation budgets, 10 reported explanation features and repeat seeds as the other notebooks. It also compares LIME perturbation dimensions of 18, 36 and 54 using the same nested feature subsets across models. Excluded features stay fixed at each case's observed values, so this tests LIME's local perturbation dimension, not retraining the model with fewer predictors. LIME's `exp.score` is a local surrogate fit score, not predictive accuracy. Runtime includes the model calls LIME makes, but excludes explainer setup and warm-up.

**Note for the write-up:** runtime is measured on each person's own machine (TabFM on this laptop's GPU, the others on CPU), so cross-model runtime differences mix model cost with hardware. Fidelity and stability don't depend on hardware.

### How long will it take?
Times one LIME-sized TabFM call and extrapolates to the full benchmark: 21 cases × 3 feature counts × 5 budgets × 5 repeats.

In [ ]:
from time import perf_counter
import controlled_lime_protocol as protocol

probe = np.tile(X_test.iloc[0].to_numpy(), (2_000, 1))
predict_proba_batched(probe[:10])            # warm-up
started = perf_counter()
predict_proba_batched(probe)
seconds_per_row = (perf_counter() - started) / len(probe)

cases = protocol.EXPLANATIONS_PER_CLASS * len(protocol.CLASS_NAMES)
total_rows = cases * len(protocol.FEATURE_COUNTS) * protocol.REPEATS * sum(protocol.SAMPLE_BUDGETS)
print(f"TabFM: {1 / seconds_per_row:,.0f} rows/second at batch size {PREDICT_BATCH_SIZE}")
print(f"Full benchmark: {total_rows:,} predictions -> roughly {total_rows * seconds_per_row / 3600:.1f} hours")

### Optional: smoke test (a few minutes)
Runs the whole pipeline on tiny budgets to catch problems before starting the multi-hour run. Nothing is saved, and the protocol settings are restored afterwards.

In [ ]:
from controlled_lime_protocol import benchmark_lime

RUN_SMOKE_TEST = True

if RUN_SMOKE_TEST:
    saved = (protocol.SAMPLE_BUDGETS, protocol.REPEATS, protocol.FEATURE_COUNTS, protocol.EXPLANATIONS_PER_CLASS)
    try:
        protocol.SAMPLE_BUDGETS, protocol.REPEATS = (100, 200), 2
        protocol.FEATURE_COUNTS, protocol.EXPLANATIONS_PER_CLASS = (18, 54), 1
        _, smoke_summary = benchmark_lime(model, X_train, X_test, y_test, predict_proba_batched,
                                          "TabFM-smoke", progress=True)
        display(smoke_summary.round(4))
    finally:
        protocol.SAMPLE_BUDGETS, protocol.REPEATS, protocol.FEATURE_COUNTS, protocol.EXPLANATIONS_PER_CLASS = saved
    print("Smoke test passed - protocol settings restored:", protocol.SAMPLE_BUDGETS, protocol.FEATURE_COUNTS)

### Full benchmark (long - resumable)
Progress prints after every (case, feature count) unit with an estimate of the time left. If it stops for any reason, run this cell again and it resumes from `tabfm_controlled_lime_checkpoint.pkl`. Delete that file only if you want to start from scratch.

In [ ]:
from IPython.display import display
from controlled_lime_protocol import evaluate_predictive_performance

model_name = "TabFM"
controlled_raw, controlled_summary = benchmark_lime(
    model, X_train, X_test, y_test, predict_proba_batched, model_name,
    checkpoint_path=f"{model_name.lower()}_controlled_lime_checkpoint.pkl",
    progress=True,
)

performance = evaluate_predictive_performance(batched_model, X_test, y_test)
print("Held-out predictive performance:", performance)
display(controlled_summary.round(4))
controlled_raw.to_csv(f"{model_name.lower()}_controlled_lime_raw.csv", index=False)
controlled_summary.to_csv(f"{model_name.lower()}_controlled_lime_summary.csv", index=False)

# Everything the methods section needs about how TabFM was run.
run_settings = {
    "context_rows": int(CONTEXT_ROWS),
    "training_rows_available": int(len(X_train)),
    "dtype": str(TABFM_DTYPE),
    "n_estimators": N_ESTIMATORS,
    "cache_context": True,
    "keep_cache_on_device": False,
    "final_predict_batch_size": int(PREDICT_BATCH_SIZE),
    "device": DEVICE,
    "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else None,
    "torch_version": torch.__version__,
    "held_out_performance": performance,
}
with open("tabfm_run_settings.json", "w") as handle:
    json.dump(run_settings, handle, indent=2)
print("Saved tabfm_run_settings.json")

## Graphs: runtime, fidelity and stability

These cells read the saved CSVs, so they work later without the GPU or re-running the benchmark. Figures are saved as PNGs in `figures/`.

**TabFM on its own:** one line per LIME perturbation dimension (18 / 36 / 54 features).

In [ ]:
from plot_controlled_results import plot_model_results, plot_model_comparison

tabfm_summary = pd.read_csv("tabfm_controlled_lime_summary.csv")
print("Saved:", plot_model_results(tabfm_summary, "TabFM", out_dir="figures"))

**All four models on the same axes.** Copy the other three `*_controlled_lime_summary.csv` files (from the LogReg, Random Forest and XGBoost notebooks) into this folder. Every summary CSV found here is included, so TabFM appears alone until the others are added.

In [ ]:
for path in plot_model_comparison(".", out_dir="figures"):
    print("Saved:", path)